Import libraries

In [13]:
import os
import pandas as pd
import numpy as np

In [14]:
import stata_setup
stata_setup.config(os.path.join('/', 'Applications', 'Stata'), 'se')
from pystata import stata

Set directory

In [15]:
cd = os.getcwd()
print(cd)

/Users/dianatagliaferri/Library/CloudStorage/OneDrive-LondonBusinessSchool/Documents/Econometrics I/ps3


**Problem 2a**

Read the dataset and perform the required manipulations

In [16]:
# Read the dataset
ps4data = pd.read_excel(os.path.join(cd, 'PS4data.xls'))

# Compute the required consumption variable
ps4data['consumption'] = (ps4data['real consumption of nondurables'] + ps4data['real consumption of services']) / ps4data['population']

ps4data.head()

,year,quarter,real consumption of nondurables,real consumption of services,cpi,real disposable income,population,equally weighted average return on the New York Stock Exchange,consumption
0,1948,1,442.3,463.8,23.533333,1055.3,102690.67,0.01414,0.008824
1,1948,2,446.9,470.5,23.933333,1087.7,102915.33,0.11755,0.008914
2,1948,3,443.3,476.6,24.466667,1107.1,103249.00,-0.09565,0.008910
3,1948,4,449.6,480.2,24.233333,1109.8,103417.67,-0.04563,0.008991
4,1949,1,451.8,482.6,23.866667,1087.8,103584.33,0.03338,0.009021


Regress consumptions on its own lags over the previous four periods and test that lags 2-4 have no predictive power

In [17]:
stata.pdataframe_to_data(ps4data[['year', 'quarter', 'consumption']], force=True)

stata_code = f'''
capture ssc install outreg2

* Generate time unique variable 
gen tq = yq(year, quarter)
format tq %tq

* Declare dataset structure as time series
tsset tq, quarterly

* Regress consumption on its own lags over the previous four periods
reg consumption L(1/4).consumption, vce(robust)

* Test that lags 2-4 have no predictive power
test L2.consumption L3.consumption L4.consumption
outreg2 using "p2a.tex", replace tex(fragment) ctitle("Consumption") dec(4)
'''

stata.run(stata_code)


. 
. capture ssc install outreg2

. 
. * Generate time unique variable 
. gen tq = yq(year, quarter)

. format tq %tq

. 
. * Declare dataset structure as time series
. tsset tq, quarterly

Time variable: tq, 1948q1 to 2002q4
        Delta: 1 quarter

. 
. * Regress consumption on its own lags over the previous four periods
. reg consumption L(1/4).consumption, vce(robust)

Linear regression                               Number of obs     =        216
                                                F(4, 211)         >   99999.00
                                                Prob > F          =     0.0000
                                                R-squared         =     0.9997
                                                Root MSE          =     7.6e-05

------------------------------------------------------------------------------
             |               Robust
 consumption | Coefficient  std. err.      t    P>|t|     [95% conf. interval]
-------------+-----------------

In [18]:
# Fix LaTeX output
with open(os.path.join(cd, 'p2a.tex'), 'r') as f: 
    full_table = f.read().replace('VARIABLES', '').replace('consumption', 'Consumption')
full_table = r'''
\begin{table}[!htbp]\centering
\caption{AR(4) Model for Consumption}
\label{tab:problem2a}
''' + '\n'+ full_table + '\n' + r'''
\end{table}
'''
with open(os.path.join(cd, 'p2a.tex'), 'w') as f: f.write(full_table)
print(full_table)


\begin{table}[!htbp]\centering
\caption{AR(4) Model for Consumption}
\label{tab:problem2a}

\begin{tabular}{lc} \hline
 & (1) \\
 & Consumption \\ \hline
 &  \\
L.Consumption & 1.1923*** \\
 & (0.0687) \\
L2.Consumption & -0.1281 \\
 & (0.1033) \\
L3.Consumption & 0.1443 \\
 & (0.1086) \\
L4.Consumption & -0.2075*** \\
 & (0.0745) \\
Constant & 0.0000 \\
 & (0.0000) \\
 &  \\
Observations & 216 \\
 R-squared & 0.9997 \\ \hline
\multicolumn{2}{c}{ Robust standard errors in parentheses} \\
\multicolumn{2}{c}{ *** p$<$0.01, ** p$<$0.05, * p$<$0.1} \\
\end{tabular}


\end{table}



**Problems 2c-2d**

Compute log variables and their ratios

In [19]:
ps4data['log_income'] = np.log(ps4data['real disposable income'] / ps4data['population'])
ps4data['log_consumption'] = np.log(ps4data['consumption'])

# Compute variables for the regression
ps4data['delta_log_income'] = ps4data['log_income'] - ps4data['log_income'].shift(1)
ps4data['delta_log_consumption'] = ps4data['log_consumption'] - ps4data['log_consumption'].shift(1)

ps4data.head()

,year,quarter,real consumption of nondurables,real consumption of services,cpi,real disposable income,population,equally weighted average return on the New York Stock Exchange,consumption,log_income,log_consumption,delta_log_income,delta_log_consumption
0,1948,1,442.3,463.8,23.533333,1055.3,102690.67,0.01414,0.008824,-4.577896,-4.730327,NaN,NaN
1,1948,2,446.9,470.5,23.933333,1087.7,102915.33,0.11755,0.008914,-4.549841,-4.720118,0.028055,0.010209
2,1948,3,443.3,476.6,24.466667,1107.1,103249.00,-0.09565,0.008910,-4.535400,-4.720634,0.014442,-0.000516
3,1948,4,449.6,480.2,24.233333,1109.8,103417.67,-0.04563,0.008991,-4.534596,-4.711562,0.000804,0.009072
4,1949,1,451.8,482.6,23.866667,1087.8,103584.33,0.03338,0.009021,-4.556229,-4.708237,-0.021633,0.003325


Run 2SLS regression

In [ ]:
stata.pdataframe_to_data(ps4data[['year', 'quarter', 'delta_log_income', 'delta_log_consumption']], force=True)

stata_code = f'''
capture ssc install outreg2

* Generate time unique variable 
gen tq = yq(year, quarter)
format tq %tq

* Declare dataset structure as time series
tsset tq, quarterly

* Regress consumption on its own lags over the previous four periods
ivregress 2sls delta_log_consumption (delta_log_income = L(2/5).delta_log_consumption), first vce(robust)
outreg2 using "p2c.tex", tex(fragment) replace dec(4)
estat firststage

* Test for endogeneity of delta_log_income
estat endogenous

* Overidentification test
estat overid 

'''

stata.run(stata_code)


. 
. capture ssc install outreg2

. 
. * Generate time unique variable 
. gen tq = yq(year, quarter)

. format tq %tq

. 
. * Declare dataset structure as time series
. tsset tq, quarterly

Time variable: tq, 1948q1 to 2002q4
        Delta: 1 quarter

. 
. * Regress consumption on its own lags over the previous four periods
. ivregress 2sls delta_log_consumption (delta_log_income = L(2/5).delta_log_con
> sumption), first 

First-stage regressions
-----------------------

                                                        Number of obs =    214
                                                        F(4, 209)     =   2.51
                                                        Prob > F      = 0.0430
                                                        R-squared     = 0.0458
                                                        Adj R-squared = 0.0276
                                                        Root MSE      = 0.0099

------------------------------------------------

In [9]:
with open(os.path.join(cd, 'p2c.tex'), 'r') as f:
    full_table = f.read().replace('VARIABLES', '').replace(r'delta\_log\_consumption', r'$\log(C_{t}/C_{t-1})$').replace(r'delta\_log\_income', r'$\log(Y_{t}/Y_{t-1})$')
full_table = r'''
\begin{table}[!htbp]\centering
\caption{Second-stage IV Regression}
\label{tab:problem2c}
''' + '\n'+ full_table + '\n' + r'''
\end{table}
'''
with open(os.path.join(cd, 'p2c.tex'), 'w') as f: f.write(full_table)
print(full_table)


\begin{table}[!htbp]\centering
\caption{Second-stage IV Regression}
\label{tab:problem2c}

\begin{tabular}{lc} \hline
 & (1) \\
 & $\log(C_{t}/C_{t-1})$ \\ \hline
 &  \\
$\log(Y_{t}/Y_{t-1})$ & 0.4369** \\
 & (0.2026) \\
Constant & 0.0026** \\
 & (0.0012) \\
 &  \\
Observations & 214 \\
 R-squared & 0.0685 \\ \hline
\multicolumn{2}{c}{ Robust standard errors in parentheses} \\
\multicolumn{2}{c}{ *** p$<$0.01, ** p$<$0.05, * p$<$0.1} \\
\end{tabular}


\end{table}



**Problem 3d**

Read the dataset and perform the required manipulations

In [10]:
sp500index = pd.read_excel(os.path.join(cd, 'SP500Index.xlsx'))

# Add column containing the log of the ratio of the index to its initial value
sp500index['log_index_ratio'] = np.log(sp500index['Level of the S&P 500 Index'] / sp500index['Level of the S&P 500 Index'].iloc[0])

sp500index.head()

,Date of Observation,Level of the S&P 500 Index,log_index_ratio
0,19600129,55.61,0.000000
1,19600229,56.12,0.009129
2,19600331,55.34,-0.004867
3,19600429,54.37,-0.022551
4,19600531,55.83,0.003948


Apply the formulas derived algebraically (conditional on $x_0$) to compute the the maximum-likelihood estimates of $\delta$ and $\sigma$

* $\hat{\delta}_{ML}=\frac{x_T}{T}$

In [11]:
delta_ml = sp500index['log_index_ratio'].iloc[-1] / (len(sp500index) - 1)
delta_ml

np.float64(0.005548394684384916)

* $\hat{\sigma}_{ML}=\sqrt{\frac{\sum_{t=1}^T(x_t-x_{t-1})^2}{T}-\frac{x_T^2}{T^2}}$

In [12]:
# Add column containing squared differences
sp500index['squared_diff'] = (sp500index['log_index_ratio'].diff())**2

# Compute sigma_ml
sigma_ml = np.sqrt(sp500index['squared_diff'].sum() / (len(sp500index) - 1) - delta_ml**2)
sigma_ml


np.float64(0.04218034607391595)